# Credit Risk Prediction

# 1. Business Objective

The objective of this project is to develop a machine learning model capable of identifying borrowers who are at higher risk of loan default. The model could support a bank's credit-risk assessment process by providing an additional data-driven indication of default risk. The analysis aims to balance the identification of genuine high-risk borrowers against the risk of incorrectly flagging borrowers who would successfully repay their loans.

This notebook explores and models loan default risk using the credit_risk_dataset.csv dataset. It covers data-quality assessment and cleaning, exploratory analysis, and comparison of Logistic Regression and Random Forest classification models. Model performance is evaluated using multiple classification metrics, ROC curves, confusion matrices and feature importance, with results interpreted in the context of credit-risk decision making.


## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/credit_risk_dataset.csv")
df.head()

## 2. Data Investigation and Cleaning

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())
print("\nMissing value %:")
print((df.isnull().mean() * 100).round(2))

In [ ]:

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDtypes:")
print(df.dtypes)

#### Duplicates

There are **165 exact duplicate rows**. These are dropped below, since keeping them would let the same applicant profile influence a model multiple times.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

#### Missing Data
Two columns have missing values:
- `person_emp_length` - 2.7% missing
- `loan_int_rate` - 9.6% missing

Both are numeric and could be missing for genuine reasons (e.g. rate not yet assigned, employment length not recorded). Median imputation is reasonable for `person_emp_length`, but `loan_int_rate` is checked against `loan_grade` first, since interest rates are usually set by grade - if so, imputing per-grade median is more accurate than a single global median.

In [ ]:
# Interest rate looks like it should track loan_grade closely
print(df.groupby("loan_grade")["loan_int_rate"].agg(["mean", "median", "count"]))

# person_emp_length: no obvious grouping column drives it - use global median
df["person_emp_length"] = df["person_emp_length"].fillna(df["person_emp_length"].median())

# Impute loan_int_rate per grade (more accurate than a single global median)
df["loan_int_rate"] = df.groupby("loan_grade")["loan_int_rate"].transform(
    lambda s: s.fillna(s.median())
)

df[["loan_int_rate", "person_emp_length"]].isnull().sum()

### Implausible values

The data contains some rows with values that can't be real such as impossible ages and Employment length longer than plausible working life given age.

In [ ]:
# Impossible ages
print("Rows with person_age > 90:")
print(df[df["person_age"] > 90][["person_age", "person_income", "person_emp_length", "loan_status"]])

# Employment length longer than plausible working life given age
implausible_emp = df[df["person_emp_length"] > (df["person_age"] - 16)]
print(f"\nRows where emp_length exceeds (age - 16): {len(implausible_emp)}")
print(implausible_emp[["person_age", "person_emp_length"]].sort_values("person_emp_length", ascending=False).head(10))

7 rows have `person_age` of 84–144 - physically impossible, almost certainly data entry errors (e.g. extra digit). 

Plus, 740 rows have `person_emp_length` implausibly high relative to `person_age` (some as extreme as 123 years of employment). Since these are a small fraction of 32K rows, dropping them is safer than trying to guess a correction.

In [ ]:
before = len(df)

df = df[df["person_age"] <= 90]
df = df[df["person_emp_length"] <= (df["person_age"] - 16)]

print(f"Dropped {before - len(df)} rows ({(before - len(df)) / before * 100:.1f}%)")
print("Remaining shape:", df.shape)

### Income outlier

In [ ]:
print(df["person_income"].describe())

sns.boxplot(x=df["person_income"])
plt.title("Person Income - Outlier Check")
plt.show()

Income is heavily right-skewed with a maximum of $6,000,000 against a 75th percentile of around $79, 000, an extreme outlier that will distort distance-based models and skew scaling. 
Instead of dropping potential, legitimate high earners, capping at a high percentile is a safer default that preserves the row while limiting its leverage.

In [ ]:
cap = df["person_income"].quantile(0.995)
print(f"Capping at 99.5th percentile: {cap:,.0f}")

n_capped = (df["person_income"] > cap).sum()
df["person_income"] = df["person_income"].clip(upper=cap)
print(f"Rows capped: {n_capped}")

## 3. Exploratory Data Analysis

### Target variable: `loan_status`
The objective is to identify borrowers who are at higher risk of loan default so the target variable, `loan_status` is investigated. The `loan_status` is 1 if the loan defaulted, 0 if repaid.

In [ ]:
print(df["loan_status"].value_counts())
print(df["loan_status"].value_counts(normalize=True).round(3))

sns.countplot(data=df, x="loan_status")
plt.title("Class Balance: loan_status")
plt.xlabel("Loan status (0 = repaid, 1 = default)")
plt.show()

**21.8% of loans in this sample defaulted** - imbalanced but manageable with:
- Stratified train/test/CV splits (essential regardless of imbalance handling)
- Class weighting in the model (`class_weight="balanced"` or equivalent)
- Resampling (SMOTE, undersampling) if a simpler model needs it

#### Relationship between loan grade and default rate

In [ ]:
grade_default = pd.crosstab(df["loan_grade"], df["loan_status"], normalize="index")
print(grade_default.round(3))

grade_default[1].sort_index().plot(kind="bar", color="firebrick")
plt.title("Default Rate by Loan Grade")
plt.ylabel("Proportion defaulted")
plt.xlabel("Loan grade")
plt.show()

Default rate climbs sharply and almost monotonically with grade: around 10% at grade A up to 98% at grade G. This is a strong, clean signal showing data is consistent.

**Caveat:** this also makes `loan_grade` a leakage risk. Loan grades are normally assigned by a lender's own risk model at origination, effectively encoding a judgment about default risk *before* it's known. Including it as a predictor risks building a model that just reconstructs the existing grading system rather than learning independently predictive signal. 

This is included and will be treated as a baseline to beat.

Alternatively, this could have been excluded, along with `loan_int_rate` (which likely derives from grade too).

### Correlation between numeric features and the target

In [ ]:
numeric_cols = ["person_age", "person_income", "person_emp_length", "loan_amnt",
                "loan_int_rate", "loan_percent_income", "cb_person_cred_hist_length", "loan_status"]

corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix (Numeric Features + Target)")
plt.show()

The two strongest predictors of `loan_status` among the numeric features are:
- **`loan_percent_income`** (r ≈ 0.38) - the loan amount as a fraction of the borrower's income. Makes sense: a loan that's a bigger bite out of income is riskier.
- **`loan_int_rate`** (r ≈ 0.34) - again plausible, though this overlaps with the `loan_grade` leakage concern above since rate is largely grade-derived.

`person_age` and `cb_person_cred_hist_length` are almost perfectly correlated with each other (r ≈ 0.86) — unsurprising, since credit history length can't exceed age by much. Only one of these is likely needed in a model; including both risks multicollinearity for little gain.

`person_income` has a modest *negative* correlation with default (r ≈ -0.14) - higher earners default less, as expected - while `loan_amnt` alone is only weakly related (r ≈ 0.11), which suggests it's the loan size *relative to income* that matters, not the raw amount.

#### Home ownership and loan intent vs. default rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ho = pd.crosstab(df["person_home_ownership"], df["loan_status"], normalize="index")[1].sort_values(ascending=False)
ho.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Default Rate by Home Ownership")
axes[0].set_ylabel("Proportion defaulted")

li = pd.crosstab(df["loan_intent"], df["loan_status"], normalize="index")[1].sort_values(ascending=False)
li.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Default Rate by Loan Intent")
axes[1].set_ylabel("Proportion defaulted")

plt.tight_layout()
plt.show()

print(ho.round(3))
print()
print(li.round(3))

Renters default at a noticeably higher rate than mortgage-holders or outright owners. Plausible, since home ownership is a rough indication for financial stability. `loan_intent` shows a smaller spread: `DEBTCONSOLIDATION` and `MEDICAL` loans default somewhat more than `VENTURE` or `EDUCATION`, though the gap here is less pronounced than for home ownership or loan grade.

## 4. Feature Engineering
### Summary of Cleaning applied:

- Dropped 165 exact duplicate rows
- Imputed `loan_int_rate` per loan grade, `person_emp_length` with the global median
- Dropped rows with impossible `person_age` (>90) or `person_emp_length` inconsistent with age
- Capped `person_income` at the 99.5th percentile

### Decisions:

1. **`loan_grade` / `loan_int_rate` leakage** included to treat as a baseline.
2. **`person_age` vs. `cb_person_cred_hist_length`** have near-duplicate information. `cb_person_cred_hist_length` had the weaker relationship with the target(loan_status) so is the safer one to drop (see below)
3. **Class imbalance (78/22)** so stratified splits and class weighting must be used as a minimum.

In [ ]:
correlations = corr.loc[["person_age", "cb_person_cred_hist_length"], "loan_status"]
strongest_column = correlations.abs().idxmax()

strongest_correlation = correlations[strongest_column]

print(correlations)
print(
    f"\nHighest correlation with loan_status: {strongest_column} "
    f"(r = {strongest_correlation:.3f})"
)
print(df.head())

# Drop the weaker of the two highly correlated features
weaker_feature = correlations.abs().idxmin()
df = df.drop(columns=[weaker_feature])

In [ ]:
# Save the cleaned dataset for use in model.ipynb
df.to_csv("data/credit_risk_dataset_cleaned.csv", index=False)
print(f"Saved cleaned dataset with shape {df.shape}")

## 5. Train/Test Split

`X` contains applicant and loan information; `y` is `loan_status`, where 1 indicates default and 0 indicates repayment.

In [ ]:
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# Load the cleaned dataframe exported from credit_risk_investigation.ipynb
df = pd.read_csv("data/credit_risk_dataset_cleaned.csv")
df.head()

In [ ]:
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print("Features:", X.shape)
print("Target:", y.shape)
print(f"Default rate: {y.mean():.1%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training default rate: {y_train.mean():.1%}")
print(f"Test default rate: {y_test.mean():.1%}")

The dataset has an imbalanced target, so `stratify=y` preserves the default rate across the train/test split.

#### Preprocessing

In [ ]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numerical_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 4. Logistic Regression

This will be the baseline model. Logistic regression was used as it is interpretable and relatively simple, appropriate for binary classification.
Note the `class_weight="balanced"` used to give greater weight to the minority class as there was an imbalance.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

In [ ]:
# Train the model
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

`predict` returns binary class labels, while `predict_proba` returns the estimated probability of default.

For credit risk, probabilities are useful because a lender may want a risk estimate rather than only a default/no-default decision.


In [ ]:
metrics = pd.DataFrame(
    {
        "metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
        "value": [
            accuracy_score(y_test, logistic_predictions),
            precision_score(y_test, logistic_predictions, zero_division=0),
            recall_score(y_test, logistic_predictions, zero_division=0),
            f1_score(y_test, logistic_predictions, zero_division=0),
            roc_auc_score(y_test, logistic_probabilities),
        ],
    }
).set_index("metric")

print(metrics.round(3))

cm = confusion_matrix(y_test, logistic_predictions)
cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Default", "Actual: Default"],
    columns=["Predicted: No Default", "Predicted: Default"]
)
print("\nConfusion matrix:")
print(cm_df)

In [ ]:
print(classification_report(y_test, logistic_predictions))

## 5. Random Forest Model

Random Forest is an ensemble of decision trees, and unlike logistic regression it can capture nonlinear relationships and interactions between features without them being specified manually. It's evaluated here as a stronger, less interpretable alternative to the logistic regression baseline.

In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)

rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

In [ ]:
print("Random Forest")

print("Accuracy:",
      round(accuracy_score(y_test, rf_predictions), 3))

print("Precision:",
      round(precision_score(y_test, rf_predictions), 3))

print("Recall:",
      round(recall_score(y_test, rf_predictions), 3))

print("F1:",
      round(f1_score(y_test, rf_predictions), 3))

print("ROC-AUC:",
      round(roc_auc_score(y_test, rf_probabilities), 3))

In [ ]:
print(classification_report(y_test, rf_predictions))

## 6. Model Comparison

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, logistic_predictions),
        accuracy_score(y_test, rf_predictions)
    ],
    "Precision": [
        precision_score(y_test, logistic_predictions),
        precision_score(y_test, rf_predictions)
    ],
    "Recall": [
        recall_score(y_test, logistic_predictions),
        recall_score(y_test, rf_predictions)
    ],
    "F1": [
        f1_score(y_test, logistic_predictions),
        f1_score(y_test, rf_predictions)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, logistic_probabilities),
        roc_auc_score(y_test, rf_probabilities)
    ]
})

results.round(3)

### Results
Overall, Random forest performaned better, with it doing better in 4 out of the 5 metrics. Although Logistic Regression achieved slightly higher recall, the Random Forest's largely higher precision indicates that its positive predictions were more reliable.

But model selection in a real credit-risk setting would depend on the relative costs of false positives and false negatives. If the main objective were to minimise missed defaults, the higher recall of Logistic Regression would be preferable, potentially after adjusting the classification threshold.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_predictions
)

plt.title("Random Forest Confusion Matrix")
plt.show()

The Random Forest correctly classified the majority of both non-defaulting and defaulting borrowers. It correctly identified 4,838 non-defaults and 1,032 defaults. However, it failed to identify 356 borrowers who subsequently defaulted, compared with 109 non-defaulting borrowers incorrectly classified as defaults. This indicates that the model is relatively conservative when identifying defaults: its predictions of default are highly reliable, but it still misses a meaningful proportion of actual defaults.

### ROC curves

The ROC curve considers many possible classification thresholds, showing how each model's true-positive rate trades off against its false-positive rate.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(
    y_test,
    logistic_probabilities,
    name="Logistic Regression",
    ax=ax
)

RocCurveDisplay.from_predictions(
    y_test,
    rf_probabilities,
    name="Random Forest",
    ax=ax
)

ax.set_title("ROC Curve Comparison")
ax.legend()
plt.show()

The ROC curves indicate that both models demonstrate strong discriminatory performance, with Logistic Regression achieving an AUC of 0.87 and Random Forest achieving an AUC of 0.93. The Random Forest curve remains closer to the top-left corner across most thresholds, indicating that it generally achieves a higher true-positive rate for a given false-positive rate. This suggests that the Random Forest is better able to distinguish between defaulting and non-defaulting borrowers.

This finding is consistent with the model comparison, where Random Forest achieved higher accuracy, precision, F1-score and ROC-AUC. However, Logistic Regression achieved slightly higher recall (0.774 compared with 0.744), meaning it identified a marginally greater proportion of actual defaults at the selected classification threshold.

## 7. Feature Importance

Feature importance analysis indicates that loan-to-income ratio, borrower income, interest rate and loan amount are the most influential features in the Random Forest model. This suggests that financial characteristics of the borrower and loan provide the majority of the predictive information used by the model. However, feature importance represents the contribution of variables to the model's predictions rather than causal relationships. Furthermore, correlated variables may distribute importance between one another. The relatively high importance of loan interest rate also warrants further investigation, as this variable may contain information related to prior credit-risk assessment.

In [ ]:
feature_names = (
    random_forest_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

In [ ]:
importances = (
    random_forest_model
    .named_steps["classifier"]
    .feature_importances_
)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(
    "importance",
    ascending=False
)

In [ ]:
feature_importance.head(15)

In [ ]:
feature_importance.head(15).sort_values(
    "importance"
).plot(
    x="feature",
    y="importance",
    kind="barh",
    figsize=(10, 6),
    legend=False
)

plt.title("Top 15 Features - Random Forest")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.show()

Feature importance shows what information the model is actually using to make its predictions. The analysis indicates that loan-to-income ratio, borrower income, interest rate and loan amount are the most influential features in the Random Forest model. This suggests that the financial characteristics of the borrower and the loan provide the majority of the predictive information used by the model. However, feature importance represents the contribution of variables to the model's predictions rather than a causal relationship, and correlated variables may split importance between one another. The relatively high importance of loan interest rate also warrants further investigation, as this variable may already encode information from a prior credit-risk assessment.

## 8. Business Interpretation

The analysis demonstrates that machine learning can provide useful predictive information for identifying borrowers at increased risk of default. Of the two models evaluated, Random Forest demonstrated the strongest overall performance, achieving an ROC-AUC of 0.931 compared with 0.873 for Logistic Regression. It also achieved substantially higher precision (90.4% versus 55.0%) and F1-score (81.6% versus 64.4%).

However, Random Forest achieved slightly lower recall than Logistic Regression (74.4% versus 77.4%), meaning that it failed to identify a greater proportion of actual defaults. In a banking context, this trade-off is important because the financial cost of a missed default may be considerably greater than the cost of incorrectly flagging a low-risk borrower. Therefore, the optimal classification threshold should ultimately be determined according to the bank's risk appetite and the relative costs of false-positive and false-negative decisions.

## 9. Conclusion
Both models are able to distinguish defaulting from non-defaulting borrowers well above chance, but they make different trade-offs. Random Forest is the stronger model overall (higher accuracy, precision, F1 and ROC-AUC), and its feature importances point to loan-to-income ratio, income, interest rate and loan amount as the main drivers of predicted risk. Logistic Regression trails on most metrics but offers a directly interpretable, coefficient-based view of risk and achieves marginally higher recall, the metric most directly tied to catching defaults before they happen.

In practice, the right model would be chosen based on the relative cost of a missed default versus a rejected good borrower, rather than on accuracy alone. Natural next steps would include tuning the classification threshold against a specific cost function, cross-validating both models rather than relying on a single train/test split, and investigating whether `loan_int_rate` is leaking information from an earlier credit decision.
